# Couple river junction node to nearest river chainage lication

This notebook demonstrates how to batch‑configure river chainage locations for river junction nodes that are missing chainage information.
The tool helps reduce manual configuration work and is especially useful when you need to configure hundreds or even thousands of river junctions.

In [1]:
import mikeplus as mp
from mikeplus.utilities.spatial_analysis_util import SpatialAnalysisUtil

db = mp.open("../tests/testdata/RiverCS/River_CS.sqlite")
db

Database<'River_CS.sqlite'>

## Run the tool

In [ ]:
df = (
        db._tables.msm_Node.select(["GeomX", "GeomY"])
        .where("TypeNo=6 AND BranchID is NULL")
        .execute()
    )
db.begin_transaction()
commit = True
try:
    for key, value in df.items():
        x = value[0]
        y = value[1]
        river_chainage = SpatialAnalysisUtil.get_nearest_river_chainage_at(
            self.db, x, y, 100.0
        )
        if river_chainage is not None:
            river = river_chainage[0]
            chainage = river_chainage[1]
            self.db._tables.msm_Node.update(
                {"BranchID": river, "BranchChainage": chainage}
            ).by_muid(key).execute()
            print(
                f"River junction of '{key}' has been coupled to '{river}' at {chainage}"
            )

except RuntimeError as e:
    print(f"An error occurred: {e}")
    commit = False

db.end_transaction(commit)

AttributeError: 'Database' object has no attribute 'BeginTransaction'

# Close database

In [ ]:
# Close the database when you're done using it.
db.close()